# Setup

In [26]:
!pip install -q streamlit pyngrok groq

In [2]:
!pip install -qU langchain langchain-community langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.2/246.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires request

# Run

In [14]:
from pyngrok import ngrok
from google.colab import userdata

# Ambil token dari Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


In [15]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

In [78]:
%%writefile streamlit_chat_app.py
# Import library yang dibutuhkan
import streamlit as st          # framework web app
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# st.title() dan st.caption() menampilkan judul dan keterangan di bagian atas
st.title("Llama Chatbot")
st.caption("Chatbot sederhana menggunakan Llama 3.3")

# Semua widget di dalam blok 'with st.sidebar:' akan muncul di panel samping
with st.sidebar:
    st.subheader("Pengaturan")

    groq_api_key = st.text_input(
        "Groq API Key",
        type="password"
    )

    reset_button = st.button(
        "Reset Percakapan",
        help="Hapus semua pesan dan mulai dari awal"
    )

# Tunggu sampai user memasukkan API key
if not groq_api_key:
    st.info(
        "Masukkan Groq API Key di sidebar untuk mulai chat.",
        icon="🗝️"
    )
    st.stop()

# Inisialisasi model hanya jika key berubah
if (
    "model" not in st.session_state
    or st.session_state.get("_last_key") != groq_api_key
):
    try:
        st.session_state._last_key = groq_api_key

        # reset history jika key berubah
        st.session_state.pop("messages", None)
        checkpointer = InMemorySaver()

        system_instruction = """
        You are SmartHomeBot, an intelligent home automation assistant.

        A human will talk to you about the status of their home devices and ask you to control them.

        You can:
        - View all available devices.
        - Turn devices on or off.
        - Change device settings when supported.
        - Check device status.
        - Reset all devices to default states.

        Always verify the requested device exists before changing it.
        If a device name is ambiguous, ask the user for clarification.

        Before executing critical actions (such as unlocking doors, disabling security systems, or resetting all devices), always confirm with the user.

        After a successful action, clearly report the new device status.

        Only interact with devices available in the smart home system.
        Do not invent devices that do not exist.
        """

        devices = {
            "living_room_light": {
                "type": "light",
                "state": "off",
                "brightness": 100
            },
            "bedroom_light": {
                "type": "light",
                "state": "off",
                "brightness": 100
            },
            "air_conditioner": {
                "type": "ac",
                "state": "off",
                "temperature": 24
            },
            "front_door_lock": {
                "type": "lock",
                "state": "locked"
            },
            "security_camera": {
                "type": "camera",
                "state": "on"
            }
        }

        def get_devices() -> str:
          """List all available smart home devices."""

          result = []

          for name, info in devices.items():
              result.append(
                  f"{name}: {info}"
              )

          return "\n".join(result)

        def get_device_status(device_name: str) -> str:
          """Get current status of a device."""

          if device_name not in devices:
              return f"Device '{device_name}' not found."

          return str(devices[device_name])

        def turn_on_device(device_name: str) -> str:
          """Turn a device on."""

          if device_name not in devices:
              return f"Device '{device_name}' not found."

          devices[device_name]["state"] = "on"

          return f"{device_name} is now ON."

        def turn_off_device(device_name: str) -> str:
          """Turn a device off."""

          if device_name not in devices:
              return f"Device '{device_name}' not found."

          devices[device_name]["state"] = "off"

          return f"{device_name} is now OFF."

        def set_ac_temperature(temp: int) -> str:
          """Set AC temperature."""

          if "air_conditioner" not in devices:
              return "AC not available."

          devices["air_conditioner"]["temperature"] = temp

          return f"AC temperature set to {temp}°C."

        def lock_door() -> str:
          """Lock the front door."""

          devices["front_door_lock"]["state"] = "locked"

          return "Front door locked."

        def unlock_door() -> str:
          """Unlock the front door."""

          devices["front_door_lock"]["state"] = "unlocked"

          return "Front door unlocked."

        def reset_home() -> str:
          """Reset all smart home devices."""

          devices["living_room_light"]["state"] = "off"
          devices["bedroom_light"]["state"] = "off"

          devices["air_conditioner"]["state"] = "off"
          devices["air_conditioner"]["temperature"] = 24

          devices["front_door_lock"]["state"] = "locked"

          return "All devices have been reset to default settings."

        # Collect all tools
        smart_home_tools = [
            get_devices,
            get_device_status,
            turn_on_device,
            turn_off_device,
            set_ac_temperature,
            lock_door,
            unlock_door,
            reset_home
        ]

        if "smart_home_agent" not in st.session_state:
          model = ChatGroq(
              model='llama-3.3-70b-versatile',
              api_key=groq_api_key,
              temperature=0.2
          )

          checkpointer = InMemorySaver()

          st.session_state.smart_home_agent = create_agent(
              model=model,
              tools=smart_home_tools,
              system_prompt=system_instruction,
              checkpointer=checkpointer
          )

    except Exception as e:
        st.error(f"Gagal membuat model Groq: {e}")
        st.stop()

# Inisialisasi list riwayat pesan kalau belum ada
# List ini menyimpan semua pesan untuk ditampilkan kembali saat skrip dijalankan ulang
if "messages" not in st.session_state:
    st.session_state.messages = []

# Kalau tombol reset diklik, hapus chat session dan riwayat pesan
if reset_button:
    st.session_state.pop("chat", None)
    st.session_state.pop("messages", None)
    # st.rerun() memaksa Streamlit me-refresh halaman dari awal
    # Ini akan menjalankan ulang seluruh skrip, dan karena session sudah dihapus,
    st.rerun()

# Loop ini menampilkan semua pesan yang sudah ada di session_state
# Setiap kali skrip dijalankan ulang, pesan-pesan ini ditampilkan kembali
for msg in st.session_state.messages:
    # st.chat_message() membuat bubble chat dengan role yang sesuai
    # role "user" = bubble di kanan, role "assistant" = bubble di kiri
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# st.chat_input() membuat kotak input di bagian bawah halaman
# Nilai yang diketik user tersimpan di variabel 'prompt'
# Variabel ini bernilai None kalau user belum mengirim pesan
prompt = st.chat_input("Ketik pesanmu di sini...")

# Hanya jalankan bagian ini kalau user mengirim pesan
if prompt:
    # Langkah 1: Tambah pesan user ke riwayat
    st.session_state.messages.append({"role": "user", "content": prompt})

    # Langkah 2: Tampilkan bubble pesan user
    with st.chat_message("user"):
        st.markdown(prompt)

    # Langkah 3: Kirim ke Gemini dan tampilkan respons
    try:
        # Kirim pesan ke Gemini melalui chat session yang sudah ada
        # chat.send_message() secara otomatis menyertakan riwayat percakapan sebelumnya
        response = st.session_state.smart_home_agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            },
            config={
                "configurable": {
                    "thread_id": "streamlit-chat"
                }
            }
        )

        answer = response["messages"][-1].content

    except Exception as e:
        # Kalau ada error (misal: rate limit, koneksi putus), tampilkan pesan error
        answer = f"Terjadi error: {e}"

    # Langkah 4: Tampilkan bubble respons assistant
    with st.chat_message("assistant"):
        st.markdown(answer)

    # Langkah 5: Simpan respons ke riwayat
    st.session_state.messages.append({"role": "assistant", "content": answer})


Overwriting streamlit_chat_app.py


In [80]:
proc = run_streamlit("streamlit_chat_app.py")

Streamlit berjalan: NgrokTunnel: "https://unmarked-grumpily-ducking.ngrok-free.dev" -> "http://localhost:8501"


In [79]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")

Streamlit dihentikan.
Tunnel ngrok ditutup.
